# Strands Multi-Agent System with AgentCore Memory (Short Term Memory) - Latest SDK

**Updated for Latest AgentCore SDK** — This notebook replaces the `MemoryManager` / `MemorySessionManager` / manual hooks pattern with the built-in `AgentCoreMemorySessionManager` from the Strands integration.

## Introduction

This notebook demonstrates how to implement a **multi-agent system with shared memory** using AWS AgentCore Memory and the Strands framework.

The original notebook (`travel-planning-agent-memory-manager.ipynb`) used the starter toolkit's `MemoryManager`, `MemorySessionManager`, and a custom `ShortTermMemoryHook` class. This updated version replaces all of that with `AgentCoreMemorySessionManager`, which handles memory operations automatically using only the core `bedrock-agentcore` SDK.

**Key migration:**
- `bedrock_agentcore_starter_toolkit.operations.memory.manager.MemoryManager` → `bedrock_agentcore.memory.MemoryClient`
- `bedrock_agentcore.memory.session.MemorySessionManager` → `AgentCoreMemorySessionManager` (Strands integration)
- Custom `ShortTermMemoryHook` class → eliminated entirely

## Tutorial Details

| Information         | Details                                                                          |
|:--------------------|:---------------------------------------------------------------------------------|
| Tutorial type       | Short Term Conversational                                                        |
| Agent usecase       | Travel Planning Assistant                                                        |
| Agentic Framework   | Strands Agents                                                                   |
| LLM model           | Anthropic Claude Haiku 4.5                                                       |
| Tutorial components | AgentCore Short-term Memory, Strands Agents, AgentCoreMemorySessionManager      |
| Example complexity  | Beginner                                                                         |

What you will learn:

- How to use `AgentCoreMemorySessionManager` for automatic memory management
- Creating specialized agents with shared memory using the latest SDK
- Implementing a coordinator agent that delegates to specialized agents
- Maintaining conversation context with minimal boilerplate code

### Scenario context

In this example, we'll create a **Travel Planning System** with:
1. A Flight Booking Assistant specialized in air travel
2. A Hotel Booking Assistant focused on accommodations
3. A Travel Coordinator that delegates to these specialized agents

## Architecture
<div style="text-align:left">
    <img src="architecture.png" width="65%" />
</div>

## Prerequisites
- Python 3.10+
- AWS account with appropriate permissions
- AWS IAM role with appropriate permissions for AgentCore Memory
- Access to Amazon Bedrock models

## Step 1: Environment Setup
Install dependencies and import the necessary libraries.

In [ ]:
!pip install -qr requirements.txt

In [ ]:
import logging
import os
from datetime import datetime

from bedrock_agentcore.memory import MemoryClient
from bedrock_agentcore.memory.integrations.strands.config import AgentCoreMemoryConfig
from bedrock_agentcore.memory.integrations.strands.session_manager import AgentCoreMemorySessionManager
from strands import Agent, tool

In [ ]:
region = os.getenv('AWS_REGION', 'us-west-2')
MODEL_ID = "global.anthropic.claude-haiku-4-5-20251001-v1:0"

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s", datefmt="%Y-%m-%d %H:%M:%S")
logger = logging.getLogger("agentcore-memory")

## Step 2: Creating Shared Memory
Create a memory resource that will be shared among our specialized agents.

The original notebook used `MemoryManager.get_or_create_memory()` from the starter toolkit. The core SDK provides `MemoryClient.create_memory()` directly.

In [ ]:
client = MemoryClient(region_name=region)

print("Creating Memory...")
memory_name = f"TravelAgent_STM_{datetime.now().strftime('%Y%m%d%H%M%S')}"

memory = client.create_memory(
    name=memory_name,
    description="Short-term memory for travel agent"
)

memory_id = memory['id']
logger.info(f"Memory ready: {memory_id}")

### Understanding Shared Memory for Multi-Agent Systems

The memory resource serves as a shared knowledge base for our travel planning system. All agents access this common memory store, enabling:

1. **Knowledge Consistency**: All agents work with the same information
2. **Context Preservation**: Conversation history is maintained across agent transitions
3. **Shared User Identity**: Both agents use the same user `actorId` — semantic search differentiates flight vs hotel context

## Step 3: Configure Actor and Session IDs

The `actorId` represents the **user** identity, not the agent. Using a consistent `actorId` ensures memory persists across sessions. Both agents share the same `actorId` and `sessionId`.

The original notebook created separate `MemorySession` objects via `MemorySessionManager.create_memory_session()`. With `AgentCoreMemorySessionManager`, this is handled automatically through `AgentCoreMemoryConfig`.

In [ ]:
# actorId represents the USER identity - consistent across sessions for memory persistence
user_actor_id = "user-001"
session_id = f"travel-session-{datetime.now().strftime('%Y%m%d%H%M%S')}"

# Both agents use the same user actorId
flight_actor_id = user_actor_id
hotel_actor_id = user_actor_id

logger.info(f"User Actor ID: {user_actor_id}")
logger.info(f"Shared Session ID: {session_id}")

## Step 4: Define System Prompts

In [ ]:
HOTEL_BOOKING_PROMPT = """You are a hotel booking assistant. Help customers find hotels, make reservations, and answer questions about accommodations and amenities.
Provide clear information about availability, pricing, and booking procedures in a friendly, helpful manner.
Keep the messages short, don't overwhelm the customer."""

FLIGHT_BOOKING_PROMPT = """You are a flight booking assistant. Help customers find flights, make reservations, and answer questions about airlines, routes, and travel policies.
Provide clear information about flight availability, pricing, schedules, and booking procedures in a friendly, helpful manner.
Keep the messages short, don't overwhelm the customer."""

## Step 5: Implementing Agent Tools

`AgentCoreMemorySessionManager` automatically handles loading conversation history, saving messages, message batching, and resource cleanup.

The original notebook required creating `MemorySession` objects, instantiating `ShortTermMemoryHook`, and passing both `hooks` and `state` to the agent. All of that is now replaced by a single `session_manager` parameter.

When `batch_size > 1`, you **must** use a `with` block or call `close()` to ensure buffered messages are flushed.

In [ ]:
@tool
def flight_booking_assistant(query: str) -> str:
    """
    Process and respond to flight booking queries.

    Args:
        query: A flight-related question about bookings, schedules, airlines, or travel policies

    Returns:
        Detailed flight information, booking options, or travel advice
    """
    try:
        flight_config = AgentCoreMemoryConfig(
            memory_id=memory_id,
            session_id=session_id,
            actor_id=flight_actor_id,
            batch_size=3
        )

        with AgentCoreMemorySessionManager(
            agentcore_memory_config=flight_config,
            region_name=region
        ) as sm:
            flight_agent = Agent(
                model=MODEL_ID,
                system_prompt=FLIGHT_BOOKING_PROMPT,
                session_manager=sm
            )
            response = flight_agent(query)
            return str(response)

    except Exception as e:
        logger.error(f"Error in flight booking assistant: {e}")
        return f"Error in flight booking assistant: {str(e)}"


@tool
def hotel_booking_assistant(query: str) -> str:
    """
    Process and respond to hotel booking queries.

    Args:
        query: A hotel-related question about accommodations, amenities, or reservations

    Returns:
        Detailed hotel information, booking options, or accommodation advice
    """
    try:
        hotel_config = AgentCoreMemoryConfig(
            memory_id=memory_id,
            session_id=session_id,
            actor_id=hotel_actor_id,
            batch_size=3
        )

        with AgentCoreMemorySessionManager(
            agentcore_memory_config=hotel_config,
            region_name=region
        ) as sm:
            hotel_agent = Agent(
                model=MODEL_ID,
                system_prompt=HOTEL_BOOKING_PROMPT,
                session_manager=sm
            )
            response = hotel_agent(query)
            return str(response)

    except Exception as e:
        logger.error(f"Error in hotel booking assistant: {e}")
        return f"Error in hotel booking assistant: {str(e)}"

## Step 6: Creating the Coordinator Agent

In [ ]:
TRAVEL_AGENT_SYSTEM_PROMPT = """
You are a comprehensive travel planning assistant that coordinates between specialized tools:
- For flight-related queries (bookings, schedules, airlines, routes) → Use the flight_booking_assistant tool
- For hotel-related queries (accommodations, amenities, reservations) → Use the hotel_booking_assistant tool
- For complete travel packages → Use both tools as needed to provide comprehensive information
- For general travel advice or simple travel questions → Answer directly

Each agent will have its own memory in case the user asks about historic data.
When handling complex travel requests, coordinate information from both tools to create a cohesive travel plan.
Provide clear organization when presenting information from multiple sources.
Keep the messages short, don't overwhelm the customer.
"""

In [ ]:
travel_agent = Agent(
    system_prompt=TRAVEL_AGENT_SYSTEM_PROMPT,
    model=MODEL_ID,
    tools=[flight_booking_assistant, hotel_booking_assistant]
)

logger.info("Travel coordinator agent created")

## Step 7: Test the Multi-Agent System

In [ ]:
response = travel_agent("Hello, I would like to book a trip from LA to Madrid. From July 1 to August 2.")
print(response)

In [ ]:
response = travel_agent("I would only like to focus on the flight at the moment. Direct flights preferred, economy class.")
print(response)

In [ ]:
response = travel_agent("Now let's look at hotels. I prefer mid-range, city center, with a pool.")
print(response)

## Step 8: Testing Memory Persistence

Create a new coordinator instance to verify the specialized agents remember previous conversations:

In [ ]:
new_travel_agent = Agent(
    system_prompt=TRAVEL_AGENT_SYSTEM_PROMPT,
    model=MODEL_ID,
    tools=[flight_booking_assistant, hotel_booking_assistant]
)

response = new_travel_agent("Can you remind me about the flights we discussed?")
print(response)

## Summary

In this notebook, we've demonstrated:

1. How to create a shared memory resource for multiple agents using `MemoryClient`
2. How to use `AgentCoreMemorySessionManager` to eliminate manual hook code
3. How to coordinate between multiple agents while maintaining conversation context
4. How memory persists across different agent instances

This approach replaces the original `MemoryManager` + `MemorySessionManager` + custom hooks pattern with a single `session_manager` parameter, reducing boilerplate significantly.

## Clean up
Delete the memory to clean up resources:

In [ ]:
# Uncomment to delete memory resource
# try:
#     client.delete_memory(memory_id=memory_id)
#     logger.info(f"Deleted memory: {memory_id}")
# except Exception as e:
#     logger.error(f"Failed to delete memory: {e}")